In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files

In [6]:
from google.colab import files
uploaded = files.upload()

Saving DatasetPenjualanToko.csv to DatasetPenjualanToko (1).csv


In [7]:
df = pd.read_csv('DatasetPenjualanToko.csv', sep=';')
df.head()

,Transaction_ID,Date,Product_Name,Category,Units_Sold,Unit_Price,Revenue,Store_Location,Payment_Method,Province,Unit Cost,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,JATNG_PURJO_00001,01/01/2023,Nabati Wafer,Snacks,7,5.0,35.000,Purworejo,Card,Jawa Tengah,4.400,NaN,NaN,NaN,NaN,NaN
1,JABAR_SUMNG_00002,01/01/2023,Sedaap Soto,Instant Noodles,16,3.8,60.800,Sumedang,Cash,Jawa Barat,3.420,NaN,NaN,NaN,NaN,NaN
2,JABAR_BANAT_00003,01/01/2023,Sari Roti Tawar,Snacks,2,12.0,24.000,Bandung Barat,Cash,Jawa Barat,10.560,NaN,NaN,NaN,NaN,NaN
3,JABAR_SUMNG_00004,01/01/2023,Pantene Shampoo 1L,Personal Care,6,45.0,270.000,Sumedang,Cash,Jawa Barat,33.750,NaN,NaN,NaN,NaN,NaN
4,JATIM_MADUN_00005,01/01/2023,Tolako Minuman Herbal,Health,16,8.5,136.000,Madiun,Cash,Jawa Timur,6.375,NaN,NaN,NaN,NaN,NaN


In [8]:
# Menghapus kolom kosong bawaan yang bernama 'Unnamed'
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Cek hasil akhirnya
display(df.head())

,Transaction_ID,Date,Product_Name,Category,Units_Sold,Unit_Price,Revenue,Store_Location,Payment_Method,Province,Unit Cost
0,JATNG_PURJO_00001,01/01/2023,Nabati Wafer,Snacks,7,5.0,35.000,Purworejo,Card,Jawa Tengah,4.400
1,JABAR_SUMNG_00002,01/01/2023,Sedaap Soto,Instant Noodles,16,3.8,60.800,Sumedang,Cash,Jawa Barat,3.420
2,JABAR_BANAT_00003,01/01/2023,Sari Roti Tawar,Snacks,2,12.0,24.000,Bandung Barat,Cash,Jawa Barat,10.560
3,JABAR_SUMNG_00004,01/01/2023,Pantene Shampoo 1L,Personal Care,6,45.0,270.000,Sumedang,Cash,Jawa Barat,33.750
4,JATIM_MADUN_00005,01/01/2023,Tolako Minuman Herbal,Health,16,8.5,136.000,Madiun,Cash,Jawa Timur,6.375


In [9]:
# Info tipe data dan jumlah non-null
df.info()

# Statistik deskriptif untuk kolom numerik
df.describe()

# Cek nilai yang hilang (missing values)
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Transaction_ID  200000 non-null  object 
 1   Date            200000 non-null  object 
 2   Product_Name    200000 non-null  object 
 3   Category        200000 non-null  object 
 4   Units_Sold      200000 non-null  int64  
 5   Unit_Price      200000 non-null  float64
 6   Revenue         200000 non-null  object 
 7   Store_Location  200000 non-null  object 
 8   Payment_Method  200000 non-null  object 
 9   Province        200000 non-null  object 
 10  Unit Cost       200000 non-null  float64
dtypes: float64(2), int64(1), object(8)
memory usage: 16.8+ MB


,0
Transaction_ID,0
Date,0
Product_Name,0
Category,0
Units_Sold,0
Unit_Price,0
Revenue,0
Store_Location,0
Payment_Method,0
Province,0


In [10]:
#mengubah tipe data kolom date menjadi format date time, pakai dayfirst = true karena data asli csv tanggalnya berawal dari hari atau format d/m/y, yang dimana date time harusnya m/d/y
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
print(df.dtypes)

Transaction_ID            object
Date              datetime64[ns]
Product_Name              object
Category                  object
Units_Sold                 int64
Unit_Price               float64
Revenue                   object
Store_Location            object
Payment_Method            object
Province                  object
Unit Cost                float64
dtype: object


**.str** untuk manipulasi string pada kolom series
**.strip()** menghapus spasi di awal dan di akhir teks, cont " Minuman " > "Minuman"
**.title()**  mengubah huruf pertama setiap kata menjadi kapital

hal ini penting agar sistem tidak salah membaca data

In [11]:
df['Category'] = df['Category'].str.strip().str.title()
df['Product_Name'] = df['Product_Name'].str.strip().str.title()
df['Province'] = df['Province'].str.strip().str.title()
df['Store_Location'] = df['Store_Location'].str.strip().str.title()

In [12]:
#mengeskstrak hari bulan tahun dan melabeli nama bulan guna kemudahan pengelompokkan data tanggal dan memudahkan saat visualisasi
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Month_Name'] = df['Date'].dt.month_name()

In [13]:
# 1. Menghapus kolom 'hantu' / Unnamed yang terbentuk dari ';;;;;'
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 2. Membersihkan format angka dari spasi dan titik, lalu ubah ke Integer
cols_to_clean = ['Unit_Price', 'Revenue', 'Unit Cost']
for col in cols_to_clean:
    df[col] = df[col].astype(str).str.replace('.', '', regex=False).str.strip().astype(int)

# 3. FEATURE ENGINEERING: Membuat kolom Profit (Sangat penting untuk Portofolio!)
df['Profit'] = df['Revenue'] - (df['Unit Cost'] * df['Units_Sold'])

# 4. Pastikan kolom Date bertipe Datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# 5. Export ke CSV yang bersih (gunakan koma standar, hilangkan index)
df.to_csv('Data_Penjualan_Looker_Ready.csv', index=False)

In [14]:
from google.colab import files
files.download('Data_Penjualan_Looker_Ready.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>